# IQL Robustness Under Distribution Shift — Full Pipeline

**CMPE 260 — Group 6 | Check-In 3**

This notebook runs the complete pipeline:
1. Install dependencies & clone repo
2. Train baseline IQL (DoubleCritic) and Q-ensemble IQL (TripleCritic)
3. Evaluate both under gravity shift and observation noise
4. Compute robustness metrics (Δ(δ), AUDC)
5. Generate comparison plots and tables

**Runtime:** ~40 min total on Colab T4 GPU (20 min per model × 2 models)

## 1. Setup & Installation

In [ ]:
# Clone the repo
!git clone https://github.com/shloakk/iql-robustness-analysis.git
%cd iql-robustness-analysis

In [ ]:
# Install dependencies
!pip install --upgrade pip
!pip install jax jaxlib flax optax
!pip install mujoco "gymnasium[mujoco]"
!pip install gym  # needed for d4rl compatibility
!pip install h5py tqdm matplotlib numpy scipy
!pip install absl-py ml_collections tensorboardX
!pip install tensorflow-probability imageio

# Install d4rl (requires gym, not gymnasium)
!pip install git+https://github.com/Farama-Foundation/d4rl@master

In [ ]:
# Verify installation
import jax
import jax.numpy as jnp
import gym
import d4rl
import numpy as np
import matplotlib.pyplot as plt

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"GPU available: {len(jax.devices('gpu')) > 0}")

## 2. Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Edit these to control the experiment
# ============================================================

ENV_NAME = 'hopper-medium-v2'  # Environment to evaluate
MAX_STEPS = 300_000            # Training steps (300k matches Check-in 2)
EVAL_EPISODES = 10             # Episodes per evaluation
EVAL_INTERVAL = 50_000         # Evaluate every N steps
SEED = 42                      # Random seed
BATCH_SIZE = 256               # Mini-batch size

# Shift configurations
GRAVITY_LEVELS = [0.5, 1.0, 1.5, 2.0]
OBS_NOISE_LEVELS = [0.0, 0.01, 0.1, 0.3]

# Hyperparameters (from configs/mujoco_config.py)
CONFIG = {
    'actor_lr': 3e-4,
    'value_lr': 3e-4,
    'critic_lr': 3e-4,
    'hidden_dims': (256, 256),
    'discount': 0.99,
    'expectile': 0.7,
    'temperature': 3.0,
    'dropout_rate': None,
    'tau': 0.005,
}

print(f"Environment: {ENV_NAME}")
print(f"Training steps: {MAX_STEPS:,}")
print(f"Gravity levels: {GRAVITY_LEVELS}")
print(f"Noise levels: {OBS_NOISE_LEVELS}")

## 3. Import Project Modules

In [ ]:
import sys
import os
import csv
from tqdm import tqdm

# Import project modules
import wrappers
from wrappers.gravity_shift import GravityShift
from wrappers.observation_noise import ObservationNoise
from dataset_utils import D4RLDataset, split_into_trajectories
from evaluation import evaluate
from learner import Learner
from common import Batch

print("All modules imported successfully.")

## 4. Dataset Loading & Normalization

In [ ]:
def normalize(dataset):
    """Normalize rewards by trajectory return range."""
    trajs = split_into_trajectories(
        dataset.observations, dataset.actions,
        dataset.rewards, dataset.masks,
        dataset.dones_float, dataset.next_observations)

    def compute_returns(traj):
        return sum(rew for _, _, rew, _, _, _ in traj)

    trajs.sort(key=compute_returns)
    dataset.rewards /= compute_returns(trajs[-1]) - compute_returns(trajs[0])
    dataset.rewards *= 1000.0


def make_env_and_dataset(env_name, seed):
    """Create environment and load D4RL dataset."""
    env = gym.make(env_name)
    env = wrappers.EpisodeMonitor(env)
    env = wrappers.SinglePrecision(env)
    env.seed(seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)

    dataset = D4RLDataset(env)
    if any(x in env_name for x in ['halfcheetah', 'walker2d', 'hopper']):
        normalize(dataset)

    return env, dataset


# Load environment and dataset
env, dataset = make_env_and_dataset(ENV_NAME, SEED)
print(f"Dataset size: {dataset.size:,} transitions")
print(f"Observation dim: {env.observation_space.shape}")
print(f"Action dim: {env.action_space.shape}")

## 5. Training Function

In [ ]:
def train_iql(env, dataset, num_critics=2, max_steps=MAX_STEPS, seed=SEED):
    """Train IQL with specified number of critics."""
    label = f"{'Baseline' if num_critics == 2 else 'Q-Ensemble'} IQL ({num_critics}Q)"
    print(f"\n{'='*60}")
    print(f"Training {label}")
    print(f"{'='*60}")

    agent = Learner(
        seed,
        env.observation_space.sample()[np.newaxis],
        env.action_space.sample()[np.newaxis],
        max_steps=max_steps,
        num_critics=num_critics,
        **CONFIG
    )

    eval_results = []

    for i in tqdm(range(1, max_steps + 1), desc=label, smoothing=0.1):
        batch = dataset.sample(BATCH_SIZE)
        info = agent.update(batch)

        if i % EVAL_INTERVAL == 0:
            stats = evaluate(agent, env, EVAL_EPISODES)
            eval_results.append({
                'step': i,
                'raw_return': stats['return'],
                'normalized_score': stats['return'],  # EpisodeMonitor normalizes
            })
            print(f"  Step {i:>7,}: return={stats['return']:.2f}")

    return agent, eval_results

## 6. Train Both Models

In [ ]:
# Train baseline IQL (DoubleCritic)
agent_2q, results_2q = train_iql(env, dataset, num_critics=2)

In [ ]:
# Train Q-ensemble IQL (TripleCritic)
agent_3q, results_3q = train_iql(env, dataset, num_critics=3)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

steps_2q = [r['step'] for r in results_2q]
scores_2q = [r['normalized_score'] for r in results_2q]
steps_3q = [r['step'] for r in results_3q]
scores_3q = [r['normalized_score'] for r in results_3q]

ax1.plot(steps_2q, scores_2q, 'b-o', label='Baseline IQL (2Q)', linewidth=2)
ax1.plot(steps_3q, scores_3q, 'r--s', label='Q-Ensemble IQL (3Q)', linewidth=2)
ax1.set_xlabel('Training Steps')
ax1.set_ylabel('Normalized Score')
ax1.set_title(f'Learning Curves — {ENV_NAME}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bar chart of final scores
final_2q = scores_2q[-1] if scores_2q else 0
final_3q = scores_3q[-1] if scores_3q else 0
bars = ax2.bar(['Baseline IQL', 'Q-Ensemble IQL'], [final_2q, final_3q],
               color=['steelblue', 'coral'])
for bar, val in zip(bars, [final_2q, final_3q]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.2f}', ha='center', fontweight='bold')
ax2.set_ylabel('Normalized Score')
ax2.set_title('Final Score Comparison')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nBaseline IQL final score: {final_2q:.2f}")
print(f"Q-Ensemble IQL final score: {final_3q:.2f}")

## 7. Evaluate Under Distribution Shift

In [ ]:
def evaluate_under_shift(agent, env_name, shift_type, shift_levels,
                         num_episodes=EVAL_EPISODES, seed=SEED):
    """Evaluate agent under distribution shift."""
    results = []
    for level in shift_levels:
        # Create shifted environment
        shifted_env = gym.make(env_name)
        if shift_type == 'gravity':
            shifted_env = GravityShift(shifted_env, gravity_scale=level)
        elif shift_type == 'obs_noise':
            shifted_env = ObservationNoise(shifted_env, noise_std=level)
        shifted_env = wrappers.EpisodeMonitor(shifted_env)
        shifted_env = wrappers.SinglePrecision(shifted_env)
        shifted_env.seed(seed)

        stats = evaluate(agent, shifted_env, num_episodes)
        results.append({
            'shift_type': shift_type,
            'shift_level': level,
            'raw_return': stats['return'],
            'episode_length': stats['length'],
        })
        print(f"  {shift_type}={level:.2f}: return={stats['return']:.2f}, "
              f"length={stats['length']:.0f}")
    return results

In [ ]:
# Evaluate baseline IQL under gravity shift
print("\n" + "="*60)
print("Evaluating Baseline IQL (2Q) under GRAVITY shift")
print("="*60)
gravity_results_2q = evaluate_under_shift(
    agent_2q, ENV_NAME, 'gravity', GRAVITY_LEVELS)

# Evaluate baseline IQL under observation noise
print("\n" + "="*60)
print("Evaluating Baseline IQL (2Q) under OBSERVATION NOISE")
print("="*60)
noise_results_2q = evaluate_under_shift(
    agent_2q, ENV_NAME, 'obs_noise', OBS_NOISE_LEVELS)

In [ ]:
# Evaluate Q-ensemble IQL under gravity shift
print("\n" + "="*60)
print("Evaluating Q-Ensemble IQL (3Q) under GRAVITY shift")
print("="*60)
gravity_results_3q = evaluate_under_shift(
    agent_3q, ENV_NAME, 'gravity', GRAVITY_LEVELS)

# Evaluate Q-ensemble IQL under observation noise
print("\n" + "="*60)
print("Evaluating Q-Ensemble IQL (3Q) under OBSERVATION NOISE")
print("="*60)
noise_results_3q = evaluate_under_shift(
    agent_3q, ENV_NAME, 'obs_noise', OBS_NOISE_LEVELS)

## 8. Compute Robustness Metrics

In [ ]:
def compute_robustness_drop(baseline_score, shifted_score):
    """Δ(δ) = (J(π, E_0) - J(π, E_δ)) / J(π, E_0)"""
    if baseline_score == 0:
        return float('inf') if shifted_score != 0 else 0.0
    return (baseline_score - shifted_score) / abs(baseline_score)


def add_robustness_metrics(results, shift_type):
    """Add Δ(δ) to each result."""
    # Find baseline (gravity=1.0 or noise=0.0)
    if shift_type == 'gravity':
        baseline = next(r['raw_return'] for r in results if r['shift_level'] == 1.0)
    else:
        baseline = next(r['raw_return'] for r in results if r['shift_level'] == 0.0)

    for r in results:
        r['robustness_drop'] = compute_robustness_drop(baseline, r['raw_return'])
        r['baseline_return'] = baseline
    return results


def compute_audc(results):
    """Area Under Degradation Curve — lower = more robust."""
    sorted_r = sorted(results, key=lambda x: x['shift_level'])
    levels = [r['shift_level'] for r in sorted_r]
    drops = [abs(r['robustness_drop']) for r in sorted_r]
    return np.trapz(drops, levels)


# Add metrics
gravity_results_2q = add_robustness_metrics(gravity_results_2q, 'gravity')
gravity_results_3q = add_robustness_metrics(gravity_results_3q, 'gravity')
noise_results_2q = add_robustness_metrics(noise_results_2q, 'obs_noise')
noise_results_3q = add_robustness_metrics(noise_results_3q, 'obs_noise')

print("Robustness metrics computed.")

## 9. Results Tables

In [ ]:
def print_comparison_table(results_2q, results_3q, shift_type):
    """Print side-by-side comparison table."""
    print(f"\n{'='*75}")
    print(f"  {shift_type.upper()} SHIFT — DoubleCritic vs TripleCritic")
    print(f"  Environment: {ENV_NAME}")
    print(f"{'='*75}")
    print(f"{'Level':<8} {'2Q Return':<12} {'2Q Δ(δ)':<10} "
          f"{'3Q Return':<12} {'3Q Δ(δ)':<10} {'Winner':<10}")
    print(f"{'-'*62}")

    for r2, r3 in zip(
        sorted(results_2q, key=lambda x: x['shift_level']),
        sorted(results_3q, key=lambda x: x['shift_level'])
    ):
        winner = '3Q ✓' if abs(r3['robustness_drop']) < abs(r2['robustness_drop']) else '2Q ✓'
        if r2['robustness_drop'] == 0 and r3['robustness_drop'] == 0:
            winner = 'Tie'
        print(f"{r2['shift_level']:<8.2f} {r2['raw_return']:<12.2f} "
              f"{r2['robustness_drop']:<10.4f} {r3['raw_return']:<12.2f} "
              f"{r3['robustness_drop']:<10.4f} {winner:<10}")

    audc_2q = compute_audc(results_2q)
    audc_3q = compute_audc(results_3q)
    print(f"\n  AUDC (lower=better): 2Q={audc_2q:.4f}, 3Q={audc_3q:.4f}")
    print(f"  Worst-case: 2Q={min(r['raw_return'] for r in results_2q):.2f}, "
          f"3Q={min(r['raw_return'] for r in results_3q):.2f}")


print_comparison_table(gravity_results_2q, gravity_results_3q, 'gravity')
print_comparison_table(noise_results_2q, noise_results_3q, 'obs_noise')

## 10. Visualization — Degradation Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Gravity: Normalized Score ---
ax = axes[0, 0]
g2 = sorted(gravity_results_2q, key=lambda x: x['shift_level'])
g3 = sorted(gravity_results_3q, key=lambda x: x['shift_level'])
ax.plot([r['shift_level'] for r in g2], [r['raw_return'] for r in g2],
        'b-o', label='Baseline IQL (2Q)', linewidth=2, markersize=8)
ax.plot([r['shift_level'] for r in g3], [r['raw_return'] for r in g3],
        'r--s', label='Q-Ensemble IQL (3Q)', linewidth=2, markersize=8)
ax.set_xlabel('Gravity Scale')
ax.set_ylabel('Normalized Score')
ax.set_title('Gravity Shift — Performance')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5, label='No shift')

# --- Gravity: Robustness Drop ---
ax = axes[0, 1]
ax.plot([r['shift_level'] for r in g2], [r['robustness_drop'] for r in g2],
        'b-o', label='Baseline IQL (2Q)', linewidth=2, markersize=8)
ax.plot([r['shift_level'] for r in g3], [r['robustness_drop'] for r in g3],
        'r--s', label='Q-Ensemble IQL (3Q)', linewidth=2, markersize=8)
ax.set_xlabel('Gravity Scale')
ax.set_ylabel('Δ(δ) — Robustness Drop')
ax.set_title('Gravity Shift — Robustness Drop')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

# --- Obs Noise: Normalized Score ---
ax = axes[1, 0]
n2 = sorted(noise_results_2q, key=lambda x: x['shift_level'])
n3 = sorted(noise_results_3q, key=lambda x: x['shift_level'])
ax.plot([r['shift_level'] for r in n2], [r['raw_return'] for r in n2],
        'b-o', label='Baseline IQL (2Q)', linewidth=2, markersize=8)
ax.plot([r['shift_level'] for r in n3], [r['raw_return'] for r in n3],
        'r--s', label='Q-Ensemble IQL (3Q)', linewidth=2, markersize=8)
ax.set_xlabel('Observation Noise σ')
ax.set_ylabel('Normalized Score')
ax.set_title('Observation Noise — Performance')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Obs Noise: Robustness Drop ---
ax = axes[1, 1]
ax.plot([r['shift_level'] for r in n2], [r['robustness_drop'] for r in n2],
        'b-o', label='Baseline IQL (2Q)', linewidth=2, markersize=8)
ax.plot([r['shift_level'] for r in n3], [r['robustness_drop'] for r in n3],
        'r--s', label='Q-Ensemble IQL (3Q)', linewidth=2, markersize=8)
ax.set_xlabel('Observation Noise σ')
ax.set_ylabel('Δ(δ) — Robustness Drop')
ax.set_title('Observation Noise — Robustness Drop')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

plt.suptitle(f'IQL Robustness Under Distribution Shift — {ENV_NAME}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('shift_evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Heatmap Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Build heatmap data
shift_labels = (
    [f'G={l}' for l in GRAVITY_LEVELS] +
    [f'N={l}' for l in OBS_NOISE_LEVELS]
)
all_2q = gravity_results_2q + noise_results_2q
all_3q = gravity_results_3q + noise_results_3q

drops_2q = [r['robustness_drop'] for r in
            sorted(gravity_results_2q, key=lambda x: x['shift_level']) +
            sorted(noise_results_2q, key=lambda x: x['shift_level'])]
drops_3q = [r['robustness_drop'] for r in
            sorted(gravity_results_3q, key=lambda x: x['shift_level']) +
            sorted(noise_results_3q, key=lambda x: x['shift_level'])]

data = np.array([drops_2q, drops_3q])

im = ax1.imshow(data, cmap='RdYlGn_r', aspect='auto', vmin=-0.5, vmax=1.0)
ax1.set_xticks(range(len(shift_labels)))
ax1.set_xticklabels(shift_labels, rotation=45, ha='right')
ax1.set_yticks([0, 1])
ax1.set_yticklabels(['Baseline (2Q)', 'Q-Ensemble (3Q)'])
ax1.set_title('Robustness Drop Δ(δ) Heatmap')

# Add text annotations
for i in range(2):
    for j in range(len(shift_labels)):
        ax1.text(j, i, f'{data[i, j]:.2f}', ha='center', va='center',
                 color='white' if abs(data[i, j]) > 0.3 else 'black', fontsize=9)

plt.colorbar(im, ax=ax1, label='Δ(δ)')

# AUDC comparison bar chart
audc_data = {
    'Gravity': (compute_audc(gravity_results_2q), compute_audc(gravity_results_3q)),
    'Obs Noise': (compute_audc(noise_results_2q), compute_audc(noise_results_3q)),
}

x = np.arange(len(audc_data))
width = 0.35
bars1 = ax2.bar(x - width/2, [v[0] for v in audc_data.values()],
                width, label='Baseline (2Q)', color='steelblue')
bars2 = ax2.bar(x + width/2, [v[1] for v in audc_data.values()],
                width, label='Q-Ensemble (3Q)', color='coral')
ax2.set_xticks(x)
ax2.set_xticklabels(audc_data.keys())
ax2.set_ylabel('AUDC (lower = more robust)')
ax2.set_title('Area Under Degradation Curve')
ax2.legend()

plt.tight_layout()
plt.savefig('robustness_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Save Results to CSV

In [ ]:
def save_results_csv(results, filename):
    """Save results to CSV."""
    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'shift_type', 'shift_level', 'raw_return',
            'episode_length', 'robustness_drop', 'baseline_return'])
        writer.writeheader()
        writer.writerows(results)
    print(f"Saved: {filename}")


# Save all results
save_results_csv(
    gravity_results_2q + noise_results_2q,
    f'results_shift_{ENV_NAME}_2Q.csv')
save_results_csv(
    gravity_results_3q + noise_results_3q,
    f'results_shift_{ENV_NAME}_3Q.csv')

# Save training curves
save_results_csv(results_2q, f'results_training_{ENV_NAME}_2Q.csv')
save_results_csv(results_3q, f'results_training_{ENV_NAME}_3Q.csv')

print("\nAll results saved. Download CSVs and PNGs for the report.")

## 13. Summary

### Key Outputs
- `training_curves.png` — Learning curves for 2Q vs 3Q
- `shift_evaluation_results.png` — Degradation curves under gravity & noise
- `robustness_heatmap.png` — Heatmap + AUDC comparison
- `results_shift_*_2Q.csv` / `results_shift_*_3Q.csv` — Raw shift data
- `results_training_*_2Q.csv` / `results_training_*_3Q.csv` — Training curves

### Next Steps
1. Run this notebook for `halfcheetah-medium-v2` and `walker2d-medium-v2`
2. Run with multiple seeds (change `SEED` to 0, 1, 2)
3. Add friction shift experiments
4. Try expectile τ ablation (change `CONFIG['expectile']` to 0.5, 0.8, 0.9)